# 🎬 AI Film Studio — ComfyUI Cloud Backend (Google Colab Free T4 GPU 15GB VRAM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onlyu66/ai-film-studio/blob/main/notebooks/comfyui_colab.ipynb)

Notebook này cho phép bạn chạy **ComfyUI trên GPU Tesla T4 (15GB VRAM) hoàn toàn miễn phí** từ Google Colab,
kết nối trực tiếp với **AI Film Studio** đang chạy trên máy tính của bạn thông qua đường hầm bảo mật Cloudflare Tunnel.

### ⚡ Hướng dẫn sử dụng nhanh trong 3 bước:
1. **Bật GPU:** Vào menu **Runtime** (Thời gian chạy) ➔ **Change runtime type** (Thay đổi loại phần cứng) ➔ Chọn **T4 GPU** ➔ Bấm **Save**.
2. **Chạy toàn bộ:** Vào menu **Runtime** ➔ **Run all** (Chạy tất cả) hoặc bấm tổ hợp phím **`Ctrl + F9`**.
3. **Kết nối:** Chờ khoảng 2 phút cho đến khi Cell cuối cùng hiện dòng thông báo màu xanh kèm link dạng:
   `https://xxxx.trycloudflare.com`
   ➔ **Copy đường link đó** dán vào ô **Server URL** trong ComfyUI Modal trên AI Film Studio (`http://localhost:8080`) và bấm **Kiểm tra**!

In [ ]:
# 1. Kiểm tra cấu hình phần cứng GPU được cấp phát bởi Google
!nvidia-smi

In [ ]:
# 2. Tải và cài đặt ComfyUI cùng ComfyUI-Manager
import os
from pathlib import Path

%cd /content
if not os.path.exists("/content/ComfyUI"):
    print("🚀 Đang tải mã nguồn ComfyUI chính thức...")
    !git clone https://github.com/comfyanonymous/ComfyUI.git
    %cd /content/ComfyUI
    print("📦 Đang cài đặt các thư viện Python...")
    !pip install -q -r requirements.txt
    
    # Cài đặt ComfyUI-Manager
    %cd /content/ComfyUI/custom_nodes
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git
    %cd /content/ComfyUI
    print("✅ Cài đặt ComfyUI & ComfyUI-Manager thành công!")
else:
    print("✅ ComfyUI đã được cài đặt sẵn.")
    %cd /content/ComfyUI

In [ ]:
# 3. Tải Checkpoint Model Điện Ảnh Siêu Tốc (SDXL Lightning - 4 Bước, 1-2 giây/ảnh)
import os

%cd /content/ComfyUI/models/checkpoints
!apt-get install -y -qq aria2

model_url = "https://huggingface.co/ByteDance/SDXL-Lightning/resolve/main/sdxl_lightning_4step_checkpoint.safetensors"
model_file = "sdxl_lightning_4step_checkpoint.safetensors"

if not os.path.exists(model_file):
    print(f"⚡ Đang tải model điện ảnh {model_file} tốc độ cao bằng aria2c...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{model_url}" -o "{model_file}"
    print("✅ Đã tải xong model Checkpoint SDXL-Lightning!")
else:
    print("✅ Model Checkpoint đã có sẵn trong thư mục.")

In [ ]:
# 4. Khởi chạy ComfyUI Server & Mở Đường Hầm Cloudflare Tunnel (Miễn phí & Bảo mật)
import subprocess
import time
import re
import os

%cd /content/ComfyUI

# Cài đặt cloudflared
if not os.path.exists("/content/cloudflared"):
    print("🌐 Đang cài đặt Cloudflare Tunnel...")
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared

# Khởi chạy ComfyUI Server chạy ngầm
print("🚀 Đang khởi động ComfyUI Server trên GPU T4 (Port 8188)...")
comfy_cmd = ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--enable-cors-header", "*"]
comfy_proc = subprocess.Popen(
    comfy_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

time.sleep(4)

# Khởi chạy Cloudflare Tunnel để đưa Port 8188 ra Internet qua HTTPS an toàn
print("🔗 Đang tạo đường hầm bảo mật TryCloudflare...")
tunnel_cmd = ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"]
tunnel_proc = subprocess.Popen(
    tunnel_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Quét tìm public URL https://*.trycloudflare.com
public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "=" * 76)
    print("🎉 COMFYUI CLOUD ĐÃ SẴN SÀNG KẾT NỐI VỚI AI FILM STUDIO!")
    print("=" * 76)
    print("👉 SAO CHÉP ĐƯỜNG LINK DƯỚI ĐÂY:")
    print(f"\n      {public_url}\n")
    print("👉 Dán vào ô 'Server URL' trong ComfyUI Modal trên AI Film Studio và bấm 'Kiểm tra'!")
    print("=" * 76 + "\n")
else:
    print("⚠️ Đang chờ tạo tunnel, vui lòng kiểm tra log bên dưới:")

# Tiếp tục in log ComfyUI quan trọng
while True:
    line = comfy_proc.stdout.readline()
    if line == "" and comfy_proc.poll() is not None:
        break
    if line:
        cleaned = line.strip()
        if any(tag in cleaned for tag in ["got prompt", "Prompt executed", "queue", "Loading", "Executing", "100%"]):
            print(f"[ComfyUI] {cleaned}")